In [ ]:
from numba import cuda
import matplotlib.pyplot as plt
import numpy as np
import time

In [ ]:
hostSrc = plt.imread('./green-lake.jpg')

SCALE = 2
hostSrc = np.hstack([
    np.vstack([
        hostSrc for _ in range(SCALE)
    ])
    for _ in range(SCALE)
])

In [ ]:
H, W, _ = hostSrc.shape
C = 3

In [ ]:
print(f"Height: {H}")
print(f"Width: {W}")

In [ ]:
hostKernel = np.array([
    [  0,   0,   1,   2,   1,   0,   0],
    [  0,   3,  13,  22,  13,   3,   0],
    [  1,  13,  59,  97,  59,  13,   1],
    [  2,  22,  97, 159,  97,  22,   2],
    [  1,  13,  59,  97,  59,  13,   1],
    [  0,   3,  13,  22,  13,   3,   0],
    [  0,   0,   1,   2,   1,   0,   0]
], np.float32)
hostKernel = hostKernel / hostKernel.sum()
KSIZE = hostKernel.shape[0]

In [ ]:
@cuda.jit
def gaussBlur(src, dst, kernel):
    # src and dst are in C, H, W shape
    c    = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    tidy = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    tidx = cuda.threadIdx.z + cuda.blockIdx.z * cuda.blockDim.z

    k = KSIZE // 2

    v = 0.0
    for dy in range(-k, k+1):
        for dx in range(-k, k+1):
            exists = ((tidx + dx) >= 0) & ((tidy + dy) >= 0)
            idx = min(max(tidx + dx, 0), W-1)
            idy = min(max(tidy + dy, 0), H-1)
            v += (exists * src[c, idy, idx]) * kernel[dy + k, dx + k]

    dst[c, tidy, tidx] = np.uint8(v)



In [ ]:
@cuda.jit
def gaussBlurShared(src, dst, kernel):
    # src and dst are in C, H, W shape
    s_kernel = cuda.shared.array((KSIZE, KSIZE), cuda.float32)
    for y in range(KSIZE):
        for x in range(KSIZE):
            s_kernel[y, x] = kernel[y, x]
    cuda.syncthreads()

    c    = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    tidy = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    tidx = cuda.threadIdx.z + cuda.blockIdx.z * cuda.blockDim.z

    k = KSIZE // 2

    v = 0.0
    for dy in range(-k, k+1):
        for dx in range(-k, k+1):
            exists = ((tidx + dx) >= 0) & ((tidy + dy) >= 0)
            idx = min(max(tidx + dx, 0), W-1)
            idy = min(max(tidy + dy, 0), H-1)
            v += (exists * src[c, idy, idx]) * kernel[dy + k, dx + k]

    dst[c, tidy, tidx] = np.uint8(v)



In [ ]:
def timeByBlockSize(blockSize: tuple[int, int], F, repeat=10):
  gridSize = (C, H//blockSize[0], W//blockSize[1])
  blockSize = (1, *blockSize)
  exeTimes = []
  for i in range(repeat):
    start_time = time.time()

    hostSrcCHW = np.moveaxis(hostSrc, -1, 0).copy()

    devSrc = cuda.to_device(hostSrcCHW)
    devDst = cuda.device_array((C, H, W), np.uint8)
    devKernel = cuda.to_device(hostKernel)

    F[gridSize, blockSize](devSrc, devDst, devKernel)

    hostDstCHW = devDst.copy_to_host()
    hostDst = np.moveaxis(hostDstCHW, 0, -1).copy()

    end_time = time.time()

    exeTimes.append(end_time - start_time)

  return sum(exeTimes) / len(exeTimes)

In [ ]:
def pair(L):
  for x in L:
    for y in L:
      yield (x, y)


power2s = list(range(3, 6))
blockSizes = list(pair([2**x for x in power2s]))
blockTicks = list(range(len(blockSizes)))

exeTimes1 = [timeByBlockSize(x, gaussBlur, 30) for x in blockSizes]
exeTimes2 = [timeByBlockSize(x, gaussBlurShared, 30) for x in blockSizes]

plt.figure(figsize=(10, 8))
plt.plot(blockTicks, exeTimes1, label = 'Without Shared Memory')
plt.plot(blockTicks, exeTimes2, label = 'With Shared Memory')
plt.xticks(blockTicks, blockSizes)

plt.ylabel('Execution Time (s)')
plt.xlabel('Block Size')

for i, v1, v2 in zip(blockTicks, exeTimes1, exeTimes2):
    plt.text(i, v1, f"{v1:.3f}", ha="center")
    plt.text(i, v2, f"{v2:.3f}", ha="center")

plt.legend()
plt.show()
